# SliceWeave — Qwen3.8-27B Oracle FFN Sparsity (Milestone 1–2)

Blueprint §9–§12, §29 + §44 Milestones 1–2 on the **primary research target: Qwen3.8-27B**.

- Memory plan combines **GPU VRAM + system RAM** (`device_map="auto"` + `max_memory`), so the model loads even when VRAM alone is insufficient.
- Designed to run **locally** (RTX 3060 6 GB VRAM + 32 GB RAM, 4-bit fallback) or **on Kaggle** (2×T4/P100, bf16) without code changes.
- If 27B weights are unavailable, set `MODEL_NAME` to any Qwen3 dense fallback (e.g. `Qwen/Qwen3-32B`).

In [ ]:
# 1. Install and import required libraries
import importlib, subprocess, sys

def _pip(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# bitsandbytes enables the 4-bit fallback path (blueprint §2: >= Q4 precision)
for mod in ["psutil", "bitsandbytes"]:
    try:
        _pip(mod.replace("-", "_"))
    except Exception as e:
        print(f"[skip] {mod}: {e}")

import gc, math, os, time
import numpy as np
import pandas as pd
import psutil
import torch
import transformers
import accelerate
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| accelerate", accelerate.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# 2. Check hardware: GPU VRAM + system RAM (blueprint §24 memory organization)
def hardware_report():
    vram_gib = 0.0
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            vram_gib += total / 1024**3
            print(f"GPU {i}: {p.name}  total {total/1024**3:.1f} GiB  free {free/1024**3:.1f} GiB")
    else:
        print("No CUDA GPU — model will run fully on CPU (slow).")
    ram = psutil.virtual_memory()
    print(f"System RAM: total {ram.total/1024**3:.1f} GiB  available {ram.available/1024**3:.1f} GiB")
    return vram_gib, ram.total / 1024**3

VRAM_GIB, RAM_GIB = hardware_report()

# 27B bf16 needs ~55 GiB; 4-bit ~16 GiB. Decide loading strategy:
#  - Kaggle (2xT4 32 GiB + 29 GiB RAM, or P100 16 GiB + 13 GiB... plus disk offload)
#  - Local 3060 (6 GiB VRAM + 32 GiB RAM) -> 4-bit required for 27B
USE_4BIT = (VRAM_GIB + RAM_GIB) < 60  # bf16 needs ~55 GiB weights + headroom
print(f"\nUSE_4BIT fallback = {USE_4BIT}")

In [ ]:
# 3. Configure model, memory budget, expert parameters (blueprint §5, §9)
MODEL_NAME = "Qwen/Qwen3.8-27B"  # TODO: replace with the exact HF repo id when published
# Fallback if the 27B id is unavailable: "Qwen/Qwen3-32B" or "Qwen/Qwen3-4B" (smoke)

GROUP_SIZE = 128                       # neurons per micro-expert (blueprint §5)
RETAIN_FRACTIONS = [1.0, 0.7, 0.5, 0.4, 0.3, 0.2, 0.15, 0.10, 0.05]
N_LAYERS_TO_TEST = 3                   # middle layers; None = all layers (Milestone 2)
PROMPTS = [
    "Debug this FastAPI JWT authentication bug: the token expires before refresh.",
    "Solve: x^2 - 5x + 6 = 0.",
    "What is the capital of Australia?",
    "Write a haiku about autumn rain.",
]
OUT_CSV = "results/csv/oracle_ffn_27b.csv"

def build_max_memory(vram_gib: float, ram_gib: float, use_4bit: bool) -> dict:
    """First fill GPU VRAM, then system RAM (blueprint §24).

    - bf16: weights must fit in VRAM+RAM (~55 GiB for 27B) -> Kaggle w/ 2xT4+RAM
    - 4-bit: ~16 GiB for 27B -> fits local 6 GiB VRAM + 32 GiB RAM
    Keep headroom for activations/KV/router overhead.
    """
    mm = {}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            mm[i] = f"{max(1, int(total - 1.5))}GiB"   # leave CUDA context + KV room
    mm["cpu"] = f"{max(2, int(ram_gib * 0.75))}GiB"    # cap CPU offload below physical RAM
    return mm

MAX_MEMORY = build_max_memory(VRAM_GIB, RAM_GIB, USE_4BIT)
print("MODEL_NAME:", MODEL_NAME)
print("max_memory plan:", MAX_MEMORY)

In [ ]:
# 4. Load Qwen3.8-27B with VRAM + system-RAM offloading
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

load_kwargs = dict(
    device_map="auto",          # overflow layers live in system RAM (blueprint §24)
    max_memory=MAX_MEMORY,      # explicit VRAM-first, RAM-second budget
)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    print("Loading in 4-bit NF4 (blueprint §2 Q4-class precision)")
else:
    # transformers 5.x renamed torch_dtype -> dtype; support both
    import inspect
    _transformers_major = int(transformers.__version__.split(".")[0])
    load_kwargs["dtype" if _transformers_major >= 5 else "torch_dtype"] = torch.bfloat16
    print("Loading in bf16")

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()
print(f"loaded in {time.time()-t0:.0f}s")

# Show how layers were split across GPU / CPU
print("\nhf_device_map summary:")
devices = {}
for mod, dev in model.hf_device_map.items():
    devices.setdefault(dev, 0)
    devices[dev] += 1
print({str(k): v for k, v in devices.items()})
print("example assignments:", dict(list(model.hf_device_map.items())[:3]))

In [ ]:
# 5. Inspect FFN modules and select target layer (blueprint §28 step 2)
def discover_ffn_layers(model):
    """{layer_key: {proj: module}} for gate/up/down projections."""
    layers = {}
    for name, module in model.named_modules():
        leaf = name.split(".")[-1]
        if leaf in ("gate_proj", "up_proj", "down_proj"):
            layer_key = ".".join(name.split(".")[:-2])  # e.g. model.layers.5
            layers.setdefault(layer_key, {})[leaf] = module
    return layers

FFN_LAYERS = discover_ffn_layers(model)
layer_ids = sorted(FFN_LAYERS, key=lambda s: int(s.rsplit(".", 1)[-1]))
n_layers = len(layer_ids)
print(f"{n_layers} transformer layers with gated FFN found")

sample = FFN_LAYERS[layer_ids[len(layer_ids)//2]]
for proj, mod in sample.items():
    print(f"  {proj}: {tuple(mod.weight.shape)}")   # expect [inter, hidden] / [hidden, inter]

# select middle layers for the single-layer test (skip embedding-flavoured early ones)
if N_LAYERS_TO_TEST is None:
    target_layers = layer_ids          # Milestone 2: all-layer sweep
else:
    idxs = [int(round(n_layers * f)) for f in (0.2, 0.5, 0.8)][:N_LAYERS_TO_TEST]
    target_layers = [layer_ids[i] for i in dict.fromkeys(idxs)]
print("target layers:", target_layers)

In [ ]:
# 6. Baseline forward pass + capture FFN intermediate z (blueprint §28 step 5)
def run_baseline(model, tokenizer, prompt: str, layer_key: str):
    """Run once; capture z = SiLU(gate(x))*up(x) at down_proj input + final logits."""
    down = model.get_submodule(f"{layer_key}.mlp.down_proj")
    captured = {}

    def hook(mod, inputs, output):
        captured["z"] = inputs[0].detach()   # [batch, seq, inter]

    handle = down.register_forward_hook(hook)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**enc)
    handle.remove()
    return {"logits": out.logits[0, -1].float().cpu(), "z": captured["z"]}

PROMPT = PROMPTS[0]
base = run_baseline(model, tokenizer, PROMPT, target_layers[0])
z = base["z"][0, -1]          # last prompt token's intermediate vector
base_logits = base["logits"]
print(f"captured z: shape={tuple(z.shape)} dtype={z.dtype}")
print(f"baseline logits: shape={tuple(base_logits.shape)}")

In [ ]:
# 7. Partition intermediate neurons into micro-experts (blueprint §5–§6)
inter = model.get_submodule(f"{target_layers[0]}.mlp.down_proj").weight.shape[1]
groups = [list(range(i, min(i + GROUP_SIZE, inter))) for i in range(0, inter, GROUP_SIZE)]
E = len(groups)
print(f"intermediate size {inter} -> {E} micro-experts of {GROUP_SIZE} neurons")

# Verify the exact decomposition: sum of expert outputs == full FFN output (§6)
down_w = model.get_submodule(f"{target_layers[0]}.mlp.down_proj").weight.float()
z_dev = z.to(down_w.device)
y_full = z_dev @ down_w.T
y_sum = torch.zeros_like(y_full)
for g in groups:
    y_sum += (z_dev[g] @ down_w[:, g].T)
decomp_err = (y_full - y_sum).abs().max().item()
print(f"decomposition check: max|y_full - sum(y_e)| = {decomp_err:.2e}  (should be ~0)")
assert decomp_err < 1e-3, "micro-expert decomposition is not exact!"

In [ ]:
# 8. Score & rank micro-experts: mean(z_e^2) and ||W_down[:,I_e] z[I_e]|| (blueprint §9)
def score_z(z_vec, groups):
    """score_e = mean(z_e^2) — cheap activation-energy score."""
    z2 = z_vec.float() ** 2
    return torch.stack([z2[g].mean() for g in groups])

def score_output(z_vec, down_weight, groups):
    """score_e = || W_down[:, I_e] z[I_e] || — true output-contribution score."""
    W = down_weight.float()          # [hidden, inter]
    scores = []
    for g in groups:
        y_e = z_vec[g].float() @ W[:, g].T
        scores.append(y_e.norm())
    return torch.stack(scores)

score_z_ = score_z(z_dev, groups)
score_out = score_output(z_dev, down_w, groups)
rank_z = torch.argsort(score_z_, descending=True)
rank_out = torch.argsort(score_out, descending=True)

print("top-5 experts by z-energy:  ", rank_z[:5].tolist())
print("top-5 experts by output-norm:", rank_out[:5].tolist())
print(f"rank agreement in top-25: {len(set(rank_z[:25].tolist()) & set(rank_out[:25].tolist()))}/25")

In [ ]:
# 9. Mask low-ranked experts & rerun forward (blueprint §28 steps 8–9)
class FFNExpertMasker:
    """Zero the intermediate activations of non-selected experts at down_proj input."""
    def __init__(self, model, layer_key: str):
        self.mlp = model.get_submodule(f"{layer_key}.mlp")
        self.handle = None
        self.keep = None

    def set_keep_mask(self, keep_mask: torch.Tensor):
        self.keep = keep_mask

    def __enter__(self):
        mlp, keep_ref = self.mlp, self

        def pre_hook(mod, args, kwargs):
            z_in = args[0] if args else kwargs.get("hidden_states")
            if keep_ref.keep is not None and z_in is not None:
                z_masked = z_in * keep_ref.keep.to(z_in.dtype).to(z_in.device)
                if args:
                    return (z_masked,) + args[1:], kwargs
                return args, {**kwargs, "hidden_states": z_masked}
            return args, kwargs

        self.handle = mlp.down_proj.register_forward_pre_hook(pre_hook, with_kwargs=True)
        return self

    def __exit__(self, *exc):
        if self.handle:
            self.handle.remove()

def keep_mask_from_rank(groups, importance, retain_fraction):
    """Boolean [inter] mask keeping the top retain_fraction of experts."""
    E = len(groups)
    k = max(1, int(round(E * retain_fraction)))
    top = torch.argsort(importance, descending=True)[:k]
    keep = torch.zeros(E, dtype=torch.bool, device=importance.device)
    keep[top] = True
    full = torch.zeros(sum(len(g) for g in groups), dtype=torch.bool,
                       device=importance.device)
    for ei, g in enumerate(groups):
        if keep[ei]:
            full[g] = True          # groups are contiguous index lists
    return full

def masked_forward(model, tokenizer, prompt, layer_key, keep_mask):
    with FFNExpertMasker(model, layer_key) as masker:
        masker.set_keep_mask(keep_mask)
        enc = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model(**enc)
    return out.logits[0, -1].float().cpu()

# sanity: keeping 100% of experts must reproduce the baseline exactly
keep_all = torch.ones(inter, dtype=torch.bool, device=z_dev.device)
check = masked_forward(model, tokenizer, PROMPT, target_layers[0], keep_all)
sanity_cos = torch.nn.functional.cosine_similarity(check, base_logits, dim=0).item()
print(f"100% retention sanity: cosine vs baseline = {sanity_cos:.6f}  (expect ~1.0)")

In [ ]:
# 10. Compare sparse vs full logits (blueprint §10 metrics)
def compute_all(sparse_logits, full_logits):
    a, b = sparse_logits.float(), full_logits.float()
    pa, pb = torch.softmax(a, -1), torch.softmax(b, -1)
    return {
        "cosine": torch.nn.functional.cosine_similarity(a, b, dim=0).item(),
        "rel_l2": ((a - b).norm() / (b.norm() + 1e-12)).item(),
        "top1_match": int(a.argmax() == b.argmax()),
        "top10_overlap": len(set(torch.topk(a, 10).indices.tolist())
                             & set(torch.topk(b, 10).indices.tolist())) / 10,
        "top50_overlap": len(set(torch.topk(a, 50).indices.tolist())
                             & set(torch.topk(b, 50).indices.tolist())) / 50,
        "kl": torch.nn.functional.kl_div(pa.log(), pb, reduction="batchmean").item(),
    }

# quick preview at two fractions using the output-aware importance
for frac in (0.5, 0.3):
    keep = keep_mask_from_rank(groups, score_out, frac)
    logits = masked_forward(model, tokenizer, PROMPT, target_layers[0], keep)
    m = compute_all(logits, base_logits)
    print(f"retain {frac:.0%}: cosine={m['cosine']:.4f} top1={m['top1_match']} "
          f"top10={m['top10_overlap']:.2f} kl={m['kl']:.4f}")

In [ ]:
# 11. Sweep retention fractions across prompts and layers -> DataFrame
rows = []
for layer_key in target_layers:
    mlp = model.get_submodule(f"{layer_key}.mlp")
    inter_l = mlp.down_proj.weight.shape[1]
    groups_l = [list(range(i, min(i + GROUP_SIZE, inter_l)))
                for i in range(0, inter_l, GROUP_SIZE)]
    for cat_i, prompt in enumerate(PROMPTS):
        base = run_baseline(model, tokenizer, prompt, layer_key)
        z_tok = base["z"][0, -1].to(mlp.down_proj.weight.device)
        base_logits_l = base["logits"]
        importance = score_output(z_tok, mlp.down_proj.weight, groups_l)
        for frac in RETAIN_FRACTIONS:
            keep = keep_mask_from_rank(groups_l, importance, frac)
            logits = masked_forward(model, tokenizer, prompt, layer_key, keep)
            m = compute_all(logits, base_logits_l)
            rows.append({"layer": layer_key, "prompt_id": cat_i,
                         "prompt": prompt[:40], "retain_fraction": frac,
                         "n_experts": len(groups_l), **m})
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"layer {layer_key} | prompt {cat_i+1}/{len(PROMPTS)} done")

df = pd.DataFrame(rows)
summary = df.groupby("retain_fraction")[["cosine", "top1_match", "top10_overlap", "kl"]].mean()
print("\n=== Mean metrics per retention fraction (all tested layers) ===")
print(summary.round(4))
df

In [ ]:
# 12. Visualize layer & fraction sensitivity (blueprint §29)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# metric-vs-fraction curves, one line per layer
for layer_key, sub in df.groupby("layer"):
    agg = sub.groupby("retain_fraction")["cosine"].mean().sort_index()
    axes[0].plot(agg.index, agg.values, marker="o", label=f"{layer_key.split('.')[-1]}")
axes[0].set_xlabel("Retention fraction (share of micro-experts kept)")
axes[0].set_ylabel("Mean logit cosine vs dense")
axes[0].set_title("Oracle sparsity by layer")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].invert_xaxis()

# layer x fraction heatmap of cosine
pivot = df.pivot_table(index="layer", columns="retain_fraction", values="cosine")
pivot = pivot.reindex(sorted(pivot.index, key=lambda s: int(s.rsplit('.', 1)[-1])))
im = axes[1].imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[1].set_xticks(range(len(pivot.columns)),
                   [f"{c:.0%}" for c in pivot.columns], rotation=45)
axes[1].set_yticks(range(len(pivot.index)),
                   [l.split(".")[-1] for l in pivot.index])
axes[1].set_title("Cosine heatmap (layer × fraction)")
axes[1].set_xlabel("Retention fraction")
axes[1].set_ylabel("Layer index")
plt.colorbar(im, ax=axes[1], label="cosine")
plt.tight_layout()
os.makedirs("results/plots", exist_ok=True)
plt.savefig("results/plots/oracle_ffn_27b.png", dpi=150)
plt.show()

In [ ]:
# 13. Save results + Blueprint §11 decision gate
os.makedirs("results/csv", exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"saved: {OUT_CSV}")

# Find the smallest retention fraction that still preserves quality
good = summary[summary["cosine"] >= 0.95].index
min_good_frac = float(good.min()) if len(good) else float("nan")
print(f"\nsmallest fraction with mean cosine >= 0.95: "
      f"{min_good_frac if min_good_frac == min_good_frac else 'none'}")

print("\n=== Blueprint §11 decision gate ===")
if min_good_frac == min_good_frac and min_good_frac <= 0.30:
    print(f"CONTINUE: {min_good_frac:.0%} experts preserve quality -> excellent signal")
elif min_good_frac == min_good_frac and min_good_frac <= 0.50:
    print(f"INVESTIGATE: {min_good_frac:.0%} needed -> interesting but uncertain")
else:
    print("WEAK SIGNAL: >50% experts required everywhere -> reconsider hypothesis")

print("\nOn Kaggle: use the right-side Output panel to download the CSV/PNG.")